# 08 · Exportar los 5 modelos para la aplicación web

La aplicación permite **clasificar con el modelo que el usuario elija** o **comparar los 5
modelos y el ensamble** en la misma foto, con las métricas de cada uno. Para eso se exportan los
5 modelos del notebook 04 a **TensorFlow Lite**, que corre con LiteRT, un intérprete liviano que
cabe en un servidor gratuito (~512 MB):

| Modelo | Tipo | Formato elegido |
|---|---|---|
| CNN propia, EfficientNetB0 | CNN | Pesos en float16 |
| ResNet50V2 | CNN | Pesos en int8 si conserva las predicciones; si no, float16 |
| ViT-B/16, DeiT-S | Transformer | Pesos en int8 si conserva las predicciones; si no, float16 |

**Por qué int8 en los tres modelos grandes:** el acelerador de CPU de LiteRT (XNNPACK) convierte
los pesos float16 a float32 al cargarlos, así que en memoria ocupan el doble que el archivo. Con
los 5 modelos en float16 (salvo ViT) el servicio llegaba a ~500 MB, al borde de los 512 MB del
servidor gratuito. Medido en el contenedor del servicio: ResNet50V2 pasa de 131 a 46 MB y DeiT-S
de 134 a 36 MB, con tiempos por imagen casi iguales. La cuantización int8 solo se acepta si
conserva al menos el 99 % de las predicciones; si no, se usa float16.

**Detalles técnicos:**
- **Normalización:** los transformers de `keras_hub` normalizan la imagen fuera del modelo
  (escala y desplazamiento por canal). Se incluye dentro del modelo exportado para que el
  servicio le entregue a todos la misma entrada: imagen RGB 224×224 en 0-255. El redimensionado
  del preprocesador se omite porque las imágenes ya llegan a 224×224 y TFLite no lo soporta.
- **Precisión:** cada modelo se reconstruye **desde su propio archivo `.keras`** del notebook
  04, cambiando en su configuración la precisión mixta por float32, y recibe los pesos entrenados
  con `load_weights` (no con `set_weights`: así la capa `Normalization` de EfficientNet
  actualiza su media y varianza internas; con `set_weights` quedaban en 0 y 1 y el modelo bajaba
  al 84 %). Antes de exportar se comprueba que el modelo reconstruido predice igual que el
  original del notebook 04.
  No se descarga nada: Kaggle no permite asociar modelos de Kaggle Models en una ejecución
  "Save & Run All", que es lo que pasaba al reconstruir los transformers desde su preset.
- **Verificación obligatoria:** cada modelo exportado se compara con su original sobre 500
  imágenes de **test** (100 por variedad). Solo se acepta si coincide en al menos el 99 % de
  las predicciones.

**Requisitos de la sesión:** Accelerator **None** (no usa GPU) y como Input el
dataset, `01-preparacion-datos`, `02_hyperband` y `04-entrenamiento-final`. Tarda unos 25 minutos.

**Salida:** `modelos/` (5 archivos `.tflite` + `modelos.json`), `modelos.zip` (la misma carpeta
comprimida), `verificacion_tflite.csv` y `muestras_test.zip` (25 fotos de test).

In [ ]:
import gc
import json
import os
import shutil
import sys
import tempfile
import time
import zipfile
from pathlib import Path

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import log_loss

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SEED = 42
N_POR_CLASE = 100
UMBRAL_COINCIDENCIA = 0.99


def find_dir_with(*names, base="/kaggle/input"):
    for root, _, files in os.walk(base):
        if all(n in files for n in names):
            return Path(root)
    raise FileNotFoundError(f"No se encontró {names}: revisa los Input del notebook")


SPLITS = find_dir_with("test.csv", "rice_data.py")
HB = find_dir_with("rice_models.py")
FINAL = find_dir_with("modelo_resnet50v2.keras", "val_y.npy")
sys.path += [str(SPLITS), str(HB)]
from rice_data import CLASSES, IMG_SIZE, find_data_root, make_dataset
from rice_models import HUB_PRESETS, MODEL_NAMES

DATA_ROOT = find_data_root()
DIR_MODELOS = OUT_DIR / "modelos"
shutil.rmtree(DIR_MODELOS, ignore_errors=True)
DIR_MODELOS.mkdir()

NOMBRES = {"cnn_propia": "CNN propia", "efficientnetb0": "EfficientNetB0", "resnet50v2": "ResNet50V2",
           "vit_b16": "ViT-B/16", "deit_s": "DeiT-S"}
CUANTIZACIONES = {m: ["int8", "fp16"] for m in ("resnet50v2", "vit_b16", "deit_s")}  # el resto: solo fp16
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

## 1. Imágenes de verificación (test)

In [ ]:
test_df = pd.read_csv(SPLITS / "test.csv")
muestra = test_df.groupby("label_idx", group_keys=False).sample(n=N_POR_CLASE, random_state=SEED).reset_index(drop=True)
X = np.concatenate([x.numpy() for x, _ in make_dataset(muestra, DATA_ROOT, batch_size=100)]).astype("float32")
y = muestra["label_idx"].to_numpy()
print("Imágenes de verificación:", X.shape)

## 2. Funciones de exportación

`envolver` arma el modelo que se exporta: entrada RGB 224×224 en 0-255 → (normalización del
transformer, si aplica) → modelo entrenado en float32.

In [ ]:
def a_float32(cfg):
    # Cambia toda política de precisión mixta de una configuración de Keras por float32.
    if isinstance(cfg, dict):
        if cfg.get("class_name") in ("DTypePolicy", "FloatDTypePolicy") and isinstance(cfg.get("config"), dict):
            return {**cfg, "config": {**cfg["config"], "name": "float32"}}
        return {k: a_float32(v) for k, v in cfg.items()}
    if isinstance(cfg, list):
        return [a_float32(v) for v in cfg]
    return "float32" if cfg == "mixed_float16" else cfg


def envolver(nombre):
    original = keras.models.load_model(FINAL / f"modelo_{nombre}.keras", compile=False)
    p_original = original.predict(X[:32], verbose=0).astype("float32")
    pesos = Path(tempfile.gettempdir()) / f"{nombre}.weights.h5"
    original.save_weights(pesos)
    keras.mixed_precision.set_global_policy("float32")
    modelo = type(original).from_config(a_float32(original.get_config()))
    modelo.load_weights(pesos)  # load_weights también actualiza el estado interno de Normalization
    del original
    pesos.unlink()
    # El modelo reconstruido en float32 debe predecir lo mismo que el original del notebook 04.
    p_float32 = modelo.predict(X[:32], verbose=0)
    # (Se compara la probabilidad y no solo la clase: con la falla de Normalization la diferencia
    # era de decenas de puntos; entre float16 y float32 es de milésimas.)
    dif_original = float(np.abs(p_float32 - p_original).max())
    assert dif_original < 0.05, f"{nombre} reconstruido no reproduce al original (dif máx. {dif_original:.3f})"
    print(f"  reconstruido en float32: dif. máx. de probabilidad con el original {dif_original:.4f}")
    entrada = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="imagen")
    x = entrada
    if nombre in HUB_PRESETS:
        ic = modelo.preprocessor.image_converter
        x = keras.layers.Rescaling(scale=np.array(ic.scale, "float32"), offset=np.array(ic.offset, "float32"))(x)
    envoltura = keras.Model(entrada, modelo(x), name=f"{nombre}_exportable")
    # La envoltura debe dar lo mismo que el modelo con su preprocesamiento normal.
    ref = modelo.predict(X[:16], verbose=0)
    dif = float(np.abs(envoltura.predict(X[:16], verbose=0) - ref).max())
    assert dif < 1e-3, f"La envoltura de {nombre} no reproduce al modelo (dif {dif})"
    return envoltura, modelo.count_params()


def convertir(envoltura, cuantizacion):
    carpeta = Path("/tmp/exportado")
    shutil.rmtree(carpeta, ignore_errors=True)
    envoltura.export(str(carpeta), verbose=False)
    conv = tf.lite.TFLiteConverter.from_saved_model(str(carpeta))
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    if cuantizacion == "fp16":
        conv.target_spec.supported_types = [tf.float16]
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]  # sin operaciones extra: LiteRT las ejecuta todas
    return conv.convert()


def predecir_tflite(contenido):
    interp = tf.lite.Interpreter(model_content=contenido)
    entrada = interp.get_input_details()[0]["index"]
    interp.resize_tensor_input(entrada, [1, IMG_SIZE, IMG_SIZE, 3])
    interp.allocate_tensors()
    salida = interp.get_output_details()[0]["index"]
    p = np.zeros((len(X), len(CLASSES)), "float32")
    interp.set_tensor(entrada, X[:1])
    interp.invoke()  # calentamiento
    t0 = time.time()
    for i in range(len(X)):
        interp.set_tensor(entrada, X[i:i + 1])
        interp.invoke()
        p[i] = interp.get_tensor(salida)[0]
    return p, (time.time() - t0) / len(X) * 1000

## 3. Exportar y verificar los 5 modelos

In [ ]:
filas, aceptados = [], {}
for nombre in MODEL_NAMES:
    print(f"\n=== {NOMBRES[nombre]} ===", flush=True)
    keras.backend.clear_session()
    gc.collect()
    try:
        envoltura, params = envolver(nombre)
        p_keras = envoltura.predict(X, batch_size=50, verbose=0)
    except Exception as e:
        print("No se pudo reconstruir:", repr(e)[:300])
        filas.append({"modelo": nombre, "estado": f"error al reconstruir: {e!r}"[:200]})
        continue
    acc_keras = float(np.mean(p_keras.argmax(1) == y))
    for cuant in CUANTIZACIONES.get(nombre, ["fp16"]):
        try:
            contenido = convertir(envoltura, cuant)
        except Exception as e:
            print(f"  {cuant}: no se pudo convertir: {repr(e)[:200]}")
            filas.append({"modelo": nombre, "cuantizacion": cuant, "estado": "error de conversión"})
            continue
        p_tfl, ms = predecir_tflite(contenido)
        coincide = float(np.mean(p_tfl.argmax(1) == p_keras.argmax(1)))
        fila = {"modelo": nombre, "cuantizacion": cuant, "MB": len(contenido) / 1e6, "ms_por_imagen_cpu": ms,
                "accuracy_keras": acc_keras, "accuracy_tflite": float(np.mean(p_tfl.argmax(1) == y)),
                "coincidencia": coincide, "dif_max_prob": float(np.abs(p_tfl - p_keras).max())}
        fila["estado"] = "aceptado" if coincide >= UMBRAL_COINCIDENCIA else "rechazado"
        filas.append(fila)
        print(f"  {cuant}: {fila['MB']:.1f} MB · coincide {coincide:.2%} · acc {fila['accuracy_tflite']:.4f} · {ms:.0f} ms/img → {fila['estado']}")
        if fila["estado"] == "aceptado":
            archivo = f"{nombre}_{cuant}.tflite"
            (DIR_MODELOS / archivo).write_bytes(contenido)
            aceptados[nombre] = {**fila, "archivo": archivo, "params_M": params / 1e6}
            break
    del envoltura
verificacion = pd.DataFrame(filas)
verificacion.to_csv(OUT_DIR / "verificacion_tflite.csv", index=False)
verificacion.round(4)

## 4. Catálogo `modelos.json`

Lo lee el servicio de inferencia. Incluye las métricas de test del notebook 04 (las que verá el
usuario en la aplicación) y los pesos del ensamble.

Los pesos del ensamble se calculan **con validación**, igual que el voto suave ponderado del
notebook 05: inverso de la log-loss de cada modelo en validación.

In [ ]:
test_res = pd.read_csv(FINAL / "test_resultados.csv").set_index("model")
y_val = np.load(FINAL / "val_y.npy")


def ll(nombre):
    p = np.clip(np.load(FINAL / f"val_probs_{nombre}.npy").astype("float64"), 1e-7, 1)
    return log_loss(y_val, p / p.sum(1, keepdims=True), labels=list(range(len(CLASSES))))


inversos = {n: 1 / ll(n) for n in aceptados}
total = sum(inversos.values())
catalogo = {
    "clases": [c.upper() for c in CLASSES],
    "tamano_entrada": IMG_SIZE,
    "rango_pixeles": "0-255 (la normalización va dentro de cada modelo)",
    "modelo_por_defecto": "resnet50v2" if "resnet50v2" in aceptados else next(iter(aceptados)),
    "modelos": [],
}
for n in MODEL_NAMES:
    if n not in aceptados:
        continue
    a = aceptados[n]
    catalogo["modelos"].append({
        "id": n,
        "nombre": NOMBRES[n],
        "tipo": "Transformer" if n in HUB_PRESETS else "CNN",
        "archivo": a["archivo"],
        "cuantizacion": a["cuantizacion"],
        "accuracy_test": round(float(test_res.loc[n, "accuracy"]), 5),
        "f1_test": round(float(test_res.loc[n, "f1_macro"]), 5),
        "params_M": round(a["params_M"], 2),
        "mb": round(a["MB"], 1),
        "ms_cpu": round(a["ms_por_imagen_cpu"], 1),
        "coincidencia_keras": round(a["coincidencia"], 4),
        "peso_ensamble": round(inversos[n] / total, 4),
    })
(DIR_MODELOS / "modelos.json").write_text(json.dumps(catalogo, indent=2, ensure_ascii=False), encoding="utf-8")
with zipfile.ZipFile(OUT_DIR / "modelos.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DIR_MODELOS.iterdir()):
        z.write(f, f.name)
faltan = [NOMBRES[n] for n in MODEL_NAMES if n not in aceptados]
print("Modelos exportados:", len(aceptados), "de 5", "| faltan:", faltan or "ninguno")
pd.DataFrame(catalogo["modelos"]).drop(columns=["archivo"])

## 5. Fotos de muestra para probar la aplicación

25 imágenes de **test** (5 por variedad), que ningún modelo vio al entrenar.

In [ ]:
dir_muestras = OUT_DIR / "muestras_test"
shutil.rmtree(dir_muestras, ignore_errors=True)
dir_muestras.mkdir()
fotos = test_df.groupby("label_idx", group_keys=False).sample(n=5, random_state=SEED + 1)
for _, fila in fotos.iterrows():
    numero = Path(fila["filepath"]).stem.split("(")[-1].rstrip(")")
    shutil.copy(DATA_ROOT / fila["filepath"], dir_muestras / f"{fila['label'].lower()}_{numero}.jpg")
with zipfile.ZipFile(OUT_DIR / "muestras_test.zip", "w") as z:
    for f in sorted(dir_muestras.iterdir()):
        z.write(f, f.name)
shutil.rmtree(dir_muestras)
print("Fotos en muestras_test.zip:", len(fotos))